In [1]:
# # Always run this FIRST
# import sys, os
# sys.path.append("/workspace")   # RunPod path
import sys
from pathlib import Path

# Find repo root by walking up until we see the "workspace" folder
p = Path.cwd().resolve()
while p != p.parent and not (p / "workspace").exists():
    p = p.parent

assert (p / "workspace").exists(), f"Couldn't find repo root above {Path.cwd().resolve()}"

REPO_ROOT = p
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Ensure package init files exist
(REPO_ROOT / "workspace" / "__init__.py").touch(exist_ok=True)
for sub in ["data", "retrieval", "models", "rlhf", "evaluation", "utils"]:
    (REPO_ROOT / "workspace" / sub / "__init__.py").touch(exist_ok=True)

print("✅ REPO_ROOT =", REPO_ROOT)
print("✅ sys.path[0] =", sys.path[0])


✅ REPO_ROOT = /workspace
✅ sys.path[0] = /workspace


In [ ]:
from pathlib import Path
import json
import os

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"


In [3]:
BASE_DIR = Path("/workspace")
ARTIFACTS = BASE_DIR / "artifacts"

DATA_DIR   = ARTIFACTS / "benchmarks"
CORPUS_DIR = ARTIFACTS / "corpus"
INDEX_DIR  = ARTIFACTS / "indexes"
MODELS_DIR = ARTIFACTS / "models"
EVAL_DIR   = ARTIFACTS / "eval_runs"

for d in [DATA_DIR, CORPUS_DIR, INDEX_DIR, MODELS_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda"


In [4]:
from workspace.data.download_dataset import download_financial_qa_10k

HF_ID = "virattt/financial-qa-10K"

download_financial_qa_10k(
    hf_id=HF_ID,
    out_dir=str(DATA_DIR),
    splits=("train",),
)

train_parquet = DATA_DIR / "virattt__financial-qa-10K_train.parquet"
test_parquet  = DATA_DIR / "virattt__financial-qa-10K_test.parquet"


2025-12-12 08:58:00,737 | data.download_dataset | INFO | Available splits for virattt/financial-qa-10K: ['train']
2025-12-12 08:58:00,739 | data.download_dataset | INFO | Downloading virattt/financial-qa-10K split=train
2025-12-12 08:58:01,309 | data.download_dataset | INFO | Saved train -> /workspace/artifacts/benchmarks/virattt__financial-qa-10K_train.parquet


In [5]:
import pandas as pd

if not test_parquet.exists():
    df = pd.read_parquet(train_parquet)
    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

    cut = int(0.8 * len(df))
    df.iloc[:cut].to_parquet(train_parquet, index=False)
    df.iloc[cut:].to_parquet(test_parquet, index=False)

print(train_parquet.exists(), test_parquet.exists())


True True


In [6]:
from workspace.data.build_benchmark import build_benchmark_from_parquet

TRAIN_JSONL = DATA_DIR / "train.jsonl"
TEST_JSONL  = DATA_DIR / "test.jsonl"

build_benchmark_from_parquet(str(train_parquet), str(TRAIN_JSONL))
build_benchmark_from_parquet(str(test_parquet),  str(TEST_JSONL))


2025-12-12 08:58:01,818 | data.build_benchmark | INFO | Saved benchmark 7000 rows -> /workspace/artifacts/benchmarks/train.jsonl
2025-12-12 08:58:01,919 | data.build_benchmark | INFO | Saved benchmark 1400 rows -> /workspace/artifacts/benchmarks/test.jsonl


In [7]:
from workspace.data.build_corpus import build_corpus_from_parquet

CHUNKS_PATH = CORPUS_DIR / "chunks.jsonl"

build_corpus_from_parquet(
    parquet_paths=[str(train_parquet), str(test_parquet)],
    out_chunks_path=str(CHUNKS_PATH),
)


2025-12-12 08:58:02,241 | data.build_corpus | INFO | Unique contexts collected: 2881
2025-12-12 08:58:02,273 | data.build_corpus | INFO | Wrote chunks: 2888 -> /workspace/artifacts/corpus/chunks.jsonl


In [8]:
from workspace.retrieval.faiss_index import build_faiss_index

META_PATH  = INDEX_DIR / "chunks_meta.jsonl"
FAISS_PATH = INDEX_DIR / "faiss_index.bin"

build_faiss_index(
    chunks_path=str(CHUNKS_PATH),
    meta_out_path=str(META_PATH),
    index_out_path=str(FAISS_PATH),
    model_name="BAAI/bge-small-en-v1.5",
    device=DEVICE,
    batch_size=64,
)


2025-12-12 08:58:05,333 | retrieval.faiss_index | INFO | Loaded chunks: 2888
2025-12-12 08:58:07,163 | retrieval.faiss_index | INFO | Embedding chunks with BAAI/bge-small-en-v1.5


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

2025-12-12 09:00:48,900 | retrieval.faiss_index | INFO | Embeddings shape: (2888, 384)
2025-12-12 09:00:48,940 | retrieval.faiss_index | INFO | Saved meta -> /workspace/artifacts/indexes/chunks_meta.jsonl
2025-12-12 09:00:48,940 | retrieval.faiss_index | INFO | Saved index -> /workspace/artifacts/indexes/faiss_index.bin


In [9]:
from workspace.retrieval.corpus_store import CorpusStore
from workspace.retrieval.faiss_index import FaissSearcher
from workspace.models.generator import OpenAIGenerator
from workspace.models.rewriter import QueryRewriter
from workspace.models.rag import AdaptiveRAG, RagConfig

store = CorpusStore(str(CHUNKS_PATH))
searcher = FaissSearcher(str(FAISS_PATH), str(META_PATH))
generator = OpenAIGenerator(model="gpt-4o-mini")
rewriter = QueryRewriter(model="gpt-4o-mini")

cfg = RagConfig(
    top_k=20,
    context_k=5,
    confidence_threshold=0.5,
    max_adaptive_steps=3,
    alpha_gen=0.6,
)

rag = AdaptiveRAG(store, searcher, generator, rewriter, cfg)


ModuleNotFoundError: No module named 'workspace.models.generator'

In [ ]:
from workspace.evaluation.eval_runner import run_eval

baseline_summary = run_eval(
    benchmark_jsonl=str(TEST_JSONL),
    answer_fn=rag.answer_baseline,
    out_path=str(EVAL_DIR / "baseline_full.json"),
    store=store,
)

adaptive_summary = run_eval(
    benchmark_jsonl=str(TEST_JSONL),
    answer_fn=rag.answer_adaptive,
    out_path=str(EVAL_DIR / "adaptive_full.json"),
    store=store,
)

baseline_summary, adaptive_summary


In [ ]:
from workspace.rlhf.train_dpo_lora import train_dpo_lora

DPO_DATA = DATA_DIR / "dpo_pairs.jsonl"
DPO_OUT  = MODELS_DIR / "dpo_lora_v1"

train_dpo_lora(
    dpo_jsonl=str(DPO_DATA),
    base_model_name="mistralai/Mistral-7B-Instruct-v0.2",
    output_dir=str(DPO_OUT),
    max_rows=200,
)


In [ ]:
rag.load_lora(str(DPO_OUT))

adaptive_dpo_summary = run_eval(
    benchmark_jsonl=str(TEST_JSONL),
    answer_fn=rag.answer_adaptive,
    out_path=str(EVAL_DIR / "adaptive_dpo_full.json"),
    store=store,
)

adaptive_dpo_summary


In [ ]:
from workspace.evaluation.presentation_examples import pick_examples

examples = pick_examples(
    eval_json_path=str(EVAL_DIR / "adaptive_full.json"),
    out_path=str(EVAL_DIR / "presentation_examples.json"),
    min_top1_gain=0.05,
    min_conf_gain=0.10,
    k=10,
)

examples["total_candidates"], len(examples["picked"])


In [ ]:
examples["picked"][0]


In [ ]:
from workspace.evaluation.error_analysis import analyze

analyze(
    eval_base_json=str(EVAL_DIR / "baseline_full.json"),
    eval_adapt_json=str(EVAL_DIR / "adaptive_full.json"),
    out_path=str(EVAL_DIR / "error_analysis.json"),
)
